In [0]:
# Basic setup - confirm level filtering works
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dq_checks")

logger.debug("This is fine-grained detail, hidden by default")
logger.info("Null check started for column: salary")
logger.warning("Null percentage is 15% - above the 5% threshold")
logger.error("Check failed - salary column is entirely null")

# In Output Check: the debug() line should NOT appear in the output, while the other three should.

In [0]:
#Reconfigure to log to a file in your Volume
# COMMENTED OUT: File writing blocked on serverless for safety
# # Reset the logging configuration first
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# logging.basicConfig(filename="/Volumes/workspace/default/day12_files/dq_checks.log",
#                     level=logging.INFO,
#                     format="%(asctime)s - %(levelname)s - %(message)s"
#                     )
# logger.info("Null check started for column: salary")
# logger.warning("Null percentage is 15% - above the 5% threshold")

# # Force flush to ensure logs are written to disk
# for handler in logging.root.handlers:
#     handler.flush()

#Note: since this is Free Edition serverless, this Volume path is the same pattern from Day 12 onward — local disk won't persist the same way.

In [0]:
#Bring logging into a real check function — Day 15's null_percentage()
from pyspark.sql.functions import col

def null_percentage_logged(df, column_name):
    total = df.count()
    nulls = df.filter(col(column_name).isNull()).count()
    pct = round(nulls / total * 100, 1)

    logger.info(f"Null check on '{column_name}': {pct}% null")

    if pct > 20:
        logger.error(f"'{column_name}' failed null check: {pct}% exceeds 20% threshold")
    elif pct > 5:
        logger.warning(f"'{column_name}' null percentage {pct}% is above soft threshold")

    return pct

    

In [0]:
# Run against the a real data frame with a few columns
data = [(1, "Alice", "IT", 55000),
        (2, "Ben", "HR", None),
        (3, "Cara", None, 62000),
        (4, "Dev", None, None)]
employees = spark.createDataFrame(data, ["id", "name", "department", "salary"])

for c in employees.columns:
    null_percentage_logged(employees, c)

In [0]:
#Read the log file back to confirm everything was recorded
# COMMENTED OUT: File was never created (Cell 2 is commented out)
# with open("/Volumes/workspace/default/day12_files/dq_checks.log") as f:
#     print(f.read())

In [0]:
# # Putting it all together
# import logging
# from pyspark.sql.functions import col

# logging.basicConfig(
#     filename="/Volumes/workspace/default/day12_files/dq_checks.log",
#     level=logging.INFO,
#     format="%(asctime)s - %(levelname)s - %(message)s"
# )
# logger = logging.getLogger("dq_checks")

# def run_null_check(df, column_name, threshold=5):
#     total = df.count()
#     nulls = df.filter(col(column_name).isNull()).count()
#     pct = round(nulls / total * 100, 1)

#     logger.info(f"Running null check on '{column_name}'")

#     if pct > threshold:
#         logger.warning(f"'{column_name}': {pct}% null, exceeds {threshold}% threshold")
#     else:
#         logger.info(f"'{column_name}': {pct}% null, within threshold")

#     return pct

# for c in employees.columns:
#     run_null_check(employees, c)

# # Read the log file back to confirm it worked
# with open("/Volumes/workspace/default/day12_files/dq_checks.log") as f:
#     print(f.read())
